<a href="https://colab.research.google.com/github/TaShapovalova/my-colab-project/blob/main/Module1_test_PYTHON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd

# 1. Находим любой файл в Colab, который заканчивается на .xlsx
excel_files = [f for f in os.listdir('.') if f.endswith('.xlsx')]

if excel_files:
    # Берем самый первый найденный Excel-файл
    actual_file_name = excel_files[0]
    print(f"Успешно найден загруженный файл: {actual_file_name}")

    # 2. Читаем его в таблицу pandas
    df = pd.read_excel(actual_file_name)

    # 3. Выводим информацию о колонках и первые строки
    print("\n--- НАЗВАНИЯ КОЛОНОК ---")
    print(df.columns.tolist())
    print("\n--- ПЕРВЫЕ 5 СТРОК ДАННЫХ ---")
    print(df.head())
else:
    print("Ошибка: Excel-файл не найден в системе. Пожалуйста, запустите ячейку выше и выберите файл повторно.")

Успешно найден загруженный файл: timeseries (1).xlsx

--- НАЗВАНИЯ КОЛОНОК ---
['Среднемесячная номинальная начисленная заработная плата работников по полному кругу организаций, Костромская область', 'ИПЦ на свежие помидоры, Бурятия', 'Уровень безработицы населения в возрасте 15-72 лет, Томская обл., процент', 'Внутренние затраты на научные исследования и разработки, млн. руб, Санкт-Петербург']

--- ПЕРВЫЕ 5 СТРОК ДАННЫХ ---
   Среднемесячная номинальная начисленная заработная плата работников по полному кругу организаций, Костромская область  \
0                                              17873                                                                      
1                                              17122                                                                      
2                                              17992                                                                      
3                                              18849                           

In [ ]:
# Посмотрим названия ВСЕХ колонок и первые строки дат
print("Все колонки:", df.columns.tolist())
print("\nПервые 5 строк всей таблицы:")
print(df.iloc[:5, :5])

Все колонки: ['Среднемесячная номинальная начисленная заработная плата работников по полному кругу организаций, Костромская область', 'ИПЦ на свежие помидоры, Бурятия', 'Уровень безработицы населения в возрасте 15-72 лет, Томская обл., процент', 'Внутренние затраты на научные исследования и разработки, млн. руб, Санкт-Петербург']

Первые 5 строк всей таблицы:
   Среднемесячная номинальная начисленная заработная плата работников по полному кругу организаций, Костромская область  \
0                                              17873                                                                      
1                                              17122                                                                      
2                                              17992                                                                      
3                                              18849                                                                      
4                      

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Вспомогательная функция для очистки ряда от пустых строк в конце
def get_clean_series(column_name):
    return df[column_name].dropna().reset_index(drop=True)

print("====== РАСЧЕТ ОТВЕТОВ ДЛЯ ТЕСТА ======\n")

# =====================================================================
# РЯД: Затраты на науку (Линейный тренд)
# =====================================================================
sci_col = 'Внутренние затраты на научные исследования и разработки, млн. руб, Санкт-Петербург'
y_sci = get_clean_series(sci_col)
X_sci = sm.add_constant(np.arange(1, len(y_sci) + 1)) # Тренд в R (tslm) начинается с 1

model_sci = sm.OLS(y_sci, X_sci).fit()

print(f"--- ВОПРОС 1 и 2 (Затраты на науку) ---")
print(f"Вопрос 1 (Коэффициент детерминации R^2): {model_sci.rsquared:.4f}")
print(f"Вопрос 2 (p-values для константы и тренда): {model_sci.pvalues.values}")
print("Если оба p-value < 0.05, то в Вопросе 2 выбирайте 'Верно', иначе 'Неверно'.\n")

print(f"--- ВОПРОС 7 (Прогноз затрат на науку) ---")
# Строим прогноз на несколько шагов вперед, чтобы вы сопоставили с 2022 годом
print("Индексы последних известных лет и прогноз:")
for i, val in enumerate(y_sci):
    print(f"Индекс {i}: {val:.1f}")
next_index = len(y_sci)
for step in range(5):
    pred = model_sci.predict([1, next_index + step])[0]
    print(f"Прогноз на следующий шаг (Индекс {next_index + step}): {pred:.1f} <-- найдите здесь 2022 год")
print("\n" + "="*50 + "\n")


# =====================================================================
# РЯД: Заработная плата (Модель Хольта-Винтерса, сезонность 12 месяцев)
# =====================================================================
wage_col = [c for c in df.columns if 'заработная плата' in c.lower()][0]
y_wage = get_clean_series(wage_col)

# Вопрос 3: Без задания параметров (авто-оптимизация)
# Используем аддитивный тренд и аддитивную сезонность, как в R HoltWinters по умолчанию
model_wage_auto = ExponentialSmoothing(y_wage, trend='add', seasonal='add', seasonal_periods=12, initialization_method='estimated').fit()
pred_wage_auto = model_wage_auto.forecast(12)

# Вопрос 8: С фиксированными параметрами
model_wage_fix = ExponentialSmoothing(y_wage, trend='add', seasonal='add', seasonal_periods=12, initialization_method='estimated').fit(smoothing_level=0.1, smoothing_trend=0.9, smoothing_seasonal=0.4)
pred_wage_fix = model_wage_fix.forecast(12)

print(f"--- ВОПРОС 3 и 8 (Заработная плата) ---")
print("Последние 3 известных значения ряда зарплат:")
print(y_wage.tail(3))
print("\nПрогноз (Вопрос 3 - авто):")
for idx, val in zip(pred_wage_auto.index, pred_wage_auto.values):
    print(f"Шаг {idx}: {val:.2f}")
print("\nПрогноз (Вопрос 8 - фиксированные параметры α=0.1, β=0.9, γ=0.4):")
for idx, val in zip(pred_wage_fix.index, pred_wage_fix.values):
    print(f"Шаг {idx}: {val:.2f}")
print("*(Сопоставьте шаги прогноза с февралем 2022 года)*\n")
print("\n" + "="*50 + "\n")


# =====================================================================
# РЯД: ИПЦ на свежие помидоры (Модель Хольта-Винтерса, сезонность 12)
# =====================================================================
cpi_col = 'ИПЦ на свежие помидоры, Бурятия'
y_cpi = get_clean_series(cpi_col)

model_cpi = ExponentialSmoothing(y_cpi, trend='add', seasonal='add', seasonal_periods=12, initialization_method='estimated').fit()
pred_cpi = model_cpi.forecast(12)

print(f"--- ВОПРОС 4 и 5 (Индекс цен / Помидоры) ---")
print(f"Вопрос 5 (Оптимальный параметр гамма / smoothing_seasonal): {model_cpi.params['smoothing_seasonal']:.4f}")
print("\nПрогноз для Вопроса 4:")
for idx, val in zip(pred_cpi.index, pred_cpi.values):
    print(f"Шаг {idx}: {val:.2f}")
print("*(Сопоставьте шаги прогноза с январем 2022 года)*\n")
print("\n" + "="*50 + "\n")


# =====================================================================
# РЯД: Уровень безработицы (Модель Хольта - тренд без сезонности)
# =====================================================================
unemp_col = [c for c in df.columns if 'безработицы' in c.lower()][0]
y_unemp = get_clean_series(unemp_col)

model_unemp = ExponentialSmoothing(y_unemp, trend='add', seasonal=None, initialization_method='estimated').fit()
pred_unemp = model_unemp.forecast(5)

print(f"--- ВОПРОС 6 (Безработица, модель Хольта) ---")
print("Последние известные значения безработицы:")
print(y_unemp.tail(3))
print("\nПрогноз на будущие годы:")
for idx, val in zip(pred_unemp.index, pred_unemp.values):
    print(f"Шаг {idx}: {val:.2f}")
print("*(Найдите шаг, соответствующий 2024 году)*\n")

====== РАСЧЕТ ОТВЕТОВ ДЛЯ ТЕСТА ======

--- ВОПРОС 1 и 2 (Затраты на науку) ---
Вопрос 1 (Коэффициент детерминации R^2): 0.9651
Вопрос 2 (p-values для константы и тренда): [1.07219918e-08 1.28579042e-08]
Если оба p-value < 0.05, то в Вопросе 2 выбирайте 'Верно', иначе 'Неверно'.

--- ВОПРОС 7 (Прогноз затрат на науку) ---
Индексы последних известных лет и прогноз:
Индекс 0: 59222.8
Индекс 1: 68990.0
Индекс 2: 84951.5
Индекс 3: 92834.4
Индекс 4: 102072.4
Индекс 5: 109711.5
Индекс 6: 114470.8
Индекс 7: 120804.0
Индекс 8: 124165.2
Индекс 9: 144851.5
Индекс 10: 135526.6
Индекс 11: 149127.2
Прогноз на следующий шаг (Индекс 12): 151676.4 <-- найдите здесь 2022 год
Прогноз на следующий шаг (Индекс 13): 159455.0 <-- найдите здесь 2022 год
Прогноз на следующий шаг (Индекс 14): 167233.7 <-- найдите здесь 2022 год
Прогноз на следующий шаг (Индекс 15): 175012.3 <-- найдите здесь 2022 год
Прогноз на следующий шаг (Индекс 16): 182790.9 <-- найдите здесь 2022 год


--- ВОПРОС 3 и 8 (Заработная плата)

In [ ]:
# Активируем встроенную в Colab поддержку R внутри Python
%load_ext rpy2.ipython

# Передаем нашу готовую таблицу df из Python в среду R
%R -i df

# --- КОД ВЫПОЛНЯЕТСЯ СТРОГО НА ЯЗЫКЕ R ---
%R colnames(df)[colnames(df) == "Среднемесячная номинальная начисленная заработная плата работников по полному кругу организаций, Костромская область"] = "wage_kolob"
%R colnames(df)[colnames(df) == "ИПЦ на свежие помидоры, Бурятия"] = "cpi_bur"
%R colnames(df)[colnames(df) == "Уровень безработицы населения в возрасте 15-72 лет, Томская обл., процент"] = "unemp_tomsk"

# Создаем временные ряды по правилам R
%R y_wage = ts(na.omit(df[["wage_kolob"]]), start=c(2013, 1), frequency=12)
%R y_cpi  = ts(na.omit(df[["cpi_bur"]]), start=c(2013, 1), frequency=12)
%R y_unemp = ts(na.omit(df[["unemp_tomsk"]]), start=c(2000, 1), frequency=1)

# ВОПРОС 3: Модель Хольта-Винтерса в R по умолчанию
%R model_wage_auto = HoltWinters(y_wage)
%R fc_wage_auto = predict(model_wage_auto, 12)
%R print("Вопрос 3 (Зарплата, февраль 2022 через R):")
%R print(fc_wage_auto[2]) # 2-й шаг — это февраль

# ВОПРОС 8: Модель Хольта-Винтерса с фиксированными параметрами в R
%R model_wage_fix = HoltWinters(y_wage, alpha=0.1, beta=0.9, gamma=0.4)
%R fc_wage_fix = predict(model_wage_fix, 12)
%R print("Вопрос 8 (Зарплата с фикс. параметрами через R):")
%R print(fc_wage_fix[2])

# ВОПРОС 4 и 5: ИПЦ Помидоры в R
%R model_cpi = HoltWinters(y_cpi)
%R fc_cpi = predict(model_cpi, 12)
%R print("Вопрос 5 (Оптимальная гамма в R):")
%R gamma_val_cpi = model_cpi$gamma
%R print(gamma_val_cpi)
%R print("Вопрос 4 (ИПЦ, январь 2022 через R):")
%R print(fc_cpi[1]) # 1-й шаг — это январь

# ВОПРОС 6: Модель Хольта (без сезонности gamma=FALSE) в R
%R model_unemp = HoltWinters(y_unemp, gamma=FALSE)
%R fc_unemp = predict(model_unemp, 5)
%R print("Вопрос 6 (Безработица на 2024 год через R):")
%R print(fc_unemp[3]) # 2022(1), 2023(2), 2024(3)

[1] "Вопрос 3 (Зарплата, февраль 2022 через R):"
[1] 34614.42
[1] "Вопрос 8 (Зарплата с фикс. параметрами через R):"
[1] 33660.17
[1] "Вопрос 5 (Оптимальная гамма в R):"
    gamma 
0.1774134 
[1] "Вопрос 4 (ИПЦ, январь 2022 через R):"
[1] 108.063
[1] "Вопрос 6 (Безработица на 2024 год через R):"
[1] 7.229282


array([7.22928187])